Dense and spare Retrival

usually we give some query and goes to blackbox where sematic search is done and that data is used as context and output is generated based on context 

1) way of improving the context we 
combine Dense and Sparse retrival 

Dense Retrival:whatever we used till now , sematic meaning using Vectors embeddings using FAISS,ChromaDB , it was doing sematic annd giving similar matching sentences 





sparse Retrival: This matches excat words using such as TF-IDF (term frequency and inverget documetn frequency ) and convert into sparse matrix at the end of the we will be able to perform exact words in vector store 
TF-IDF,BM25 and other general Embedding techniques (Exact Keyword Search)


Hybrid Search technique

Sematic search(Dense Matrix) and Sparse Matrix ( Exact Match) :::---- combining both of them  

using these statergy we can get better context of words 

using it mathmatically : 1) dense Retrival *(embeddings+ cosine similairty)
                         2) spare retrival( TF-IDF)

score of hybrid :- alpha * score of dense + (1-alpha)* score of sparse

score of dense:- cosine similairty between the input and output vectorstore

score of sparse:- TF-IDF(input,vectorrepresentation)

Alpha = weightage of sparse or dense 


Lets say : build application using LLM
            cosine similairy between the sentence1: "langchain helps build LLM apps" and query " build application using LLm" will be high 
            but according to TF-TDF only matching its comparitively low coz(its exact search)
            
            if i apply =0.5 
                    the context will be better 

In [2]:
### Hybrid search Technique Implementation 

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.retrievers import BM25Retriever
from langchain.schema import Document 

In [3]:
Docs=[
    Document(page_content="langchain helps building LLM Applications"),
    Document(page_content="pinecone is a vector database for sematic search"),
    Document(page_content="TThe Eiffel Tower is Located in Paris"),
    Document(page_content="Langchian can be used to develop agentic AI Applications")
]
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
dense_vectors =FAISS.from_documents(Docs,embedding_model)
dense_retriever = dense_vectors.as_retriever()

d:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
from langchain.retrievers import EnsembleRetriever

#### Sparse Retriver(BM25)
##### BM25 is a ranking function used in information retrival to asses the relevance of documents to a search query, its a probabilistic model that considers term frequency 

### Spare Retiver 
sparse_retriever =  BM25Retriever.from_documents(Docs)
sparse_retriever.k=3

### Step 4: combine with Ensemble Retriver

Hybrid_retiever= EnsembleRetriever(
	retrievers=[dense_retriever, sparse_retriever],
	weights=[0.6, 0.4]
)

In [15]:
Hybrid_retiever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000247CFC39410>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x0000024780A23750>, k=3)], weights=[0.6, 0.4])

In [16]:
## Step 5 : query and get results 

query = "how can i buld an application using LLM?"
results = Hybrid_retiever.invoke(query )

# step 6 : print results
for i , doc in enumerate(results):
    print(f"\n Document{i+1}:\n{doc.page_content}")


 Document1:
Langchian can be used to develop agentic AI Applications

 Document2:
pinecone is a vector database for sematic search

 Document3:
TThe Eiffel Tower is Located in Paris

 Document4:
langchain helps building LLM Applications


In [18]:
## Rag Pipeline with hybrid Retriever 
from langchain.chat_models import init_chat_model
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain

In [19]:
prompts = PromptTemplate.from_template("""
                                       Answer the question based on the context below 
                                       context{context}
                                       question:{input}""")
llm= init_chat_model("openai:gpt-3.5-turbo",temperature=0.2)
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000024784C3DB10>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000024784D16010>, root_client=<openai.OpenAI object at 0x0000024784813F50>, root_async_client=<openai.AsyncOpenAI object at 0x0000024784D15D10>, temperature=0.2, model_kwargs={}, openai_api_key=SecretStr('**********'))

In [21]:
document_chain= create_stuff_documents_chain(llm=llm,prompt=prompts)

## create full Rag chain

rag_chain = create_retrieval_chain(retriever=Hybrid_retiever,combine_docs_chain=document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000247CFC39410>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x0000024780A23750>, k=3)], weights=[0.6, 0.4]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n                                       Answer the question based on the context below \n                                       c

In [22]:
query={"input":"how can i build a LLM"}
response =rag_chain.invoke(query)

#step10:Ouput

print("Answer:\n",response["answer"])

print("\n Source documents")
for i , doc in enumerate(response["context"]):
    print(f"\nDoc {i+1}: {doc.page_content}")

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}